# Farmer (TwoStageFarmer) — Analytic Linear-Region Boundary Planes (3D)

This notebook derives and **plots the analytic boundary planes** where the second-stage recourse decisions change regime
for the classic *farmer* model implemented in `farmer_problem.py`.

We use the 3-scenario yield multipliers used in your `run_farmer_case.py`:  
- good: $y=1.2$  
- fair: $y=1.0$  
- bad:  $y=0.8$  
with equal probability.

Feasible region (first-stage planting acres) is the tetrahedron:
\[
x_w\ge 0,\; x_c\ge 0,\; x_b\ge 0,\; x_w+x_c+x_b \le 500.
\]
We also visualize the **budget face** $x_w+x_c+x_b=500$.


## 1) Analytic kink (boundary) planes

From `TwoStageFarmer` in `farmer_problem.py`, each scenario LP has:

- wheat minimum requirement:  
  \[
  2.5\,y\,x_w + y_w - w_w \ge 200,
  \]
  buy cost 238, sell price 170.

- corn minimum requirement:
  \[
  3.0\,y\,x_c + y_c - w_c \ge 240,
  \]
  buy cost 210, sell price 150.

- beets selling with a quota $T=6000$ at favorable price 36 and the remainder at price 10:
  \[
  w_{bf} + w_{bu} \le 20\,y\,x_b,\qquad w_{bf}\le 6000.
  \]

Because **buy prices exceed sell prices** (wheat: 238 > 170; corn: 210 > 150), the optimal recourse never buys a crop just to resell it.  
So each crop's optimal second-stage policy is:

- wheat:
  - if $2.5y x_w \ge 200$ then buy 0 and sell surplus,
  - else buy the deficit and sell 0.
  - kink at \(2.5y x_w = 200 \Rightarrow x_w = \frac{200}{2.5y} = \frac{80}{y}\).

- corn:
  - kink at \(3y x_c = 240 \Rightarrow x_c = \frac{240}{3y} = \frac{80}{y}\).

- beets:
  - kink at \(20y x_b = 6000 \Rightarrow x_b = \frac{6000}{20y} = \frac{300}{y}\).

Thus, for each scenario multiplier \(y\), the **boundary planes** are:
\[
x_w=\frac{80}{y},\qquad x_c=\frac{80}{y},\qquad x_b=\frac{300}{y}.
\]

With \(y\in\{1.2,1.0,0.8\}\):
- wheat/corn planes at \(x=\{66.\overline{6},\,80,\,100\}\),
- beets planes at \(x_b=\{250,\,300,\,375\}\).


In [ ]:
import numpy as np
import plotly.graph_objects as go

TOTAL = 500.0

# scenario yield multipliers (as used in run_farmer_case.py)
ys = [1.2, 1.0, 0.8]
names = ["good (1.2)", "fair (1.0)", "bad (0.8)"]

# kink thresholds
thr_wc = [80.0/y for y in ys]       # wheat and corn share the same numeric thresholds here
thr_b  = [300.0/y for y in ys]      # beets quota threshold

thr_wc, thr_b


In [ ]:
def tetra_edges(total=500.0):
    # vertices of tetrahedron x>=0, y>=0, z>=0, x+y+z<=total
    V = np.array([
        [0,0,0],
        [total,0,0],
        [0,total,0],
        [0,0,total],
    ], dtype=float)
    edges = [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
    return V, edges

def add_tetra_wireframe(fig, total=500.0, name="feasible tetrahedron"):
    V, edges = tetra_edges(total)
    for (i,j) in edges:
        x, y, z = zip(V[i], V[j])
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode="lines",
            name=name if (i,j)==edges[0] else None,
            showlegend=(i,j)==edges[0],
            line=dict(width=4),
            opacity=0.55
        ))
    return fig

def add_budget_face(fig, total=500.0, name="budget face (xw+xc+xb=500)", opacity=0.18):
    # triangle vertices
    V = np.array([[total,0,0],[0,total,0],[0,0,total]], dtype=float)
    i, j, k = [0], [1], [2]
    fig.add_trace(go.Mesh3d(
        x=V[:,0], y=V[:,1], z=V[:,2],
        i=i, j=j, k=k,
        name=name,
        opacity=opacity,
        showlegend=True
    ))
    return fig

def plane_triangle(axis, t, total=500.0):
    # returns 3 vertices of intersection of plane axis=t with tetrahedron
    # axis in {"w","c","b"} corresponding to x_w, x_c, x_b
    t = float(t)
    if t < -1e-12 or t > total + 1e-12:
        return None
    t = min(max(t, 0.0), total)
    if axis == "w":
        return np.array([[t,0,0],[t,total-t,0],[t,0,total-t]], dtype=float)
    if axis == "c":
        return np.array([[0,t,0],[total-t,t,0],[0,t,total-t]], dtype=float)
    if axis == "b":
        return np.array([[0,0,t],[total-t,0,t],[0,total-t,t]], dtype=float)
    raise ValueError("axis must be 'w','c','b'")

def add_plane(fig, axis, t, label, opacity=0.22):
    tri = plane_triangle(axis, t, TOTAL)
    if tri is None:
        return fig
    fig.add_trace(go.Mesh3d(
        x=tri[:,0], y=tri[:,1], z=tri[:,2],
        i=[0], j=[1], k=[2],
        name=label,
        opacity=opacity,
        showlegend=True
    ))
    return fig

fig = go.Figure()
fig = add_tetra_wireframe(fig, TOTAL)
fig = add_budget_face(fig, TOTAL)

# add boundary planes
for y, nm, t in zip(ys, names, thr_wc):
    add_plane(fig, "w", t, f"wheat kink: xw=80/{y:.1f}={t:.2f} ({nm})")
for y, nm, t in zip(ys, names, thr_wc):
    add_plane(fig, "c", t, f"corn kink: xc=80/{y:.1f}={t:.2f} ({nm})")
for y, nm, t in zip(ys, names, thr_b):
    add_plane(fig, "b", t, f"beets quota kink: xb=300/{y:.1f}={t:.2f} ({nm})")

fig.update_layout(
    title="Farmer analytic boundary planes (linear-region kinks) in the feasible tetrahedron",
    scene=dict(
        xaxis_title="wheat acres (xw)",
        yaxis_title="corn acres (xc)",
        zaxis_title="beets acres (xb)",
        aspectmode="data"
    ),
    legend=dict(itemsizing="constant")
)

fig


## 2) (Optional) Boundary lines restricted to the budget face

On the budget face \(x_b = 500 - x_w - x_c\), each plane becomes a line segment:

- wheat kink plane \(x_w=t\) becomes the line \(x_w=t\) inside the triangle.
- corn kink plane \(x_c=t\) becomes the line \(x_c=t\) inside the triangle.
- beets kink plane \(x_b=t\) becomes \(x_w + x_c = 500 - t\).

The cell below draws these **analytic** line segments directly on the budget face.


In [ ]:
import plotly.graph_objects as go
import numpy as np

def budget_face_line_w(t, total=500.0):
    # xw=t, xc in [0, total-t], xb = total - t - xc
    a = np.array([t, 0.0, total - t])
    b = np.array([t, total - t, 0.0])
    return a, b

def budget_face_line_c(t, total=500.0):
    a = np.array([0.0, t, total - t])
    b = np.array([total - t, t, 0.0])
    return a, b

def budget_face_line_b(t, total=500.0):
    # xb=t => xw+xc = total - t with xw,xc>=0
    s = total - t
    a = np.array([0.0, s, t])
    b = np.array([s, 0.0, t])
    return a, b

fig2 = go.Figure()
fig2 = add_tetra_wireframe(fig2, TOTAL)
fig2 = add_budget_face(fig2, TOTAL, opacity=0.25)

def add_line(fig, p, q, name):
    fig.add_trace(go.Scatter3d(
        x=[p[0], q[0]], y=[p[1], q[1]], z=[p[2], q[2]],
        mode="lines",
        name=name,
        line=dict(width=7),
        opacity=0.9
    ))
    return fig

for y, nm, t in zip(ys, names, thr_wc):
    p,q = budget_face_line_w(t, TOTAL)
    add_line(fig2, p,q, f"on face: xw={t:.2f} ({nm})")

for y, nm, t in zip(ys, names, thr_wc):
    p,q = budget_face_line_c(t, TOTAL)
    add_line(fig2, p,q, f"on face: xc={t:.2f} ({nm})")

for y, nm, t in zip(ys, names, thr_b):
    p,q = budget_face_line_b(t, TOTAL)
    add_line(fig2, p,q, f"on face: xb={t:.2f} ({nm})")

fig2.update_layout(
    title="Analytic boundary lines (kinks) restricted to the budget face",
    scene=dict(
        xaxis_title="wheat acres (xw)",
        yaxis_title="corn acres (xc)",
        zaxis_title="beets acres (xb)",
        aspectmode="data"
    )
)
fig2
